# Author: Tahjae Jackson 
# Description: This notebook serves to do preliminary analysis/ pre-processing of the cell-count datasheet

In [1]:
#  importing libraries needed

import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    SelectKBest, 
    chi2, 
    mutual_info_classif,
    RFE,
    RFECV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

In [4]:
#  Loading the data 
df = pd.read_csv("datasheets/cell-count.csv")
cols = df.columns
print(cols)

#  cehcking what columns have missing cells

# Count missing values in each column
missing_counts = df.isna().sum()

# Show only columns with missing values
print(missing_counts[missing_counts > 0])


Index(['project', 'subject', 'condition', 'age', 'sex', 'treatment',
       'response', 'sample', 'sample_type', 'time_from_treatment_start',
       'b_cell', 'cd8_t_cell', 'cd4_t_cell', 'nk_cell', 'monocyte'],
      dtype='object')
response    1422
dtype: int64


It is seen that the response column has some vacant cells. Some further assessment needs to be done to see if there is a correlation to other columns. Upon initial inspection of the dataset, it is hypothesized that there is relationship between the treatment column and the response column.

In [ ]:
print(df["treatment"].value_counts(dropna=False))
print(df["response"].value_counts(dropna=False))



treatment
miraclib     4695
phauximab    4383
none         1422
Name: count, dtype: int64
response
yes    4611
no     4467
NaN    1422
Name: count, dtype: int64


As seen above, you can see that there is a correlation between the none in the treatment column and the empty cells in the response column.

In [8]:

missing_response = df[df["response"].isna()]
print(missing_response[["subject", "treatment", "response"]].head())
print(
    pd.crosstab(
        df["treatment"],
        df["response"].isna(),
        dropna=False
    )
)

   subject treatment response
30  sbj010      none      NaN
31  sbj010      none      NaN
32  sbj010      none      NaN
36  sbj012      none      NaN
37  sbj012      none      NaN
response   False  True 
treatment              
miraclib    4695      0
none           0   1422
phauximab   4383      0


Checking what columns have unique entries 

In [11]:
for col in cols:
    duplicates = df[df[col].duplicated()]
    print("Duplicate", col," IDs:", len(duplicates))
    # print(duplicates)

Duplicate project  IDs: 10497
Duplicate subject  IDs: 7000
Duplicate condition  IDs: 10497
Duplicate age  IDs: 10470
Duplicate sex  IDs: 10498
Duplicate treatment  IDs: 10497
Duplicate response  IDs: 10497
Duplicate sample  IDs: 0
Duplicate sample_type  IDs: 10498
Duplicate time_from_treatment_start  IDs: 10497
Duplicate b_cell  IDs: 3594
Duplicate cd8_t_cell  IDs: 2685
Duplicate cd4_t_cell  IDs: 2439
Duplicate nk_cell  IDs: 3147
Duplicate monocyte  IDs: 2858


As seen, the sample table does not have any duplicate entries and can exist as a primary key

In [13]:
print("Rows:", len(df))
print("Unique subjects:", df["subject"].nunique())
print("Unique samples:", df["sample"].nunique())
subject_counts = df.groupby("subject").size()
print(subject_counts.describe())
print(subject_counts.value_counts().sort_index())

Rows: 10500
Unique subjects: 3500
Unique samples: 10500
count    3500.0
mean        3.0
std         0.0
min         3.0
25%         3.0
50%         3.0
75%         3.0
max         3.0
dtype: float64
3    3500
Name: count, dtype: int64


Exploratory analysis showed that the dataset contains 10,500 samples from 3,500 unique subjects. Each sample identifier is unique, making sample_id an appropriate primary key for the samples table. Additionally, each subject contributes exactly three samples, indicating a one-to-many relationship between subjects and samples. To avoid duplication of subject-level metadata, a separate subjects table should be created with subject_id as the primary key and referenced from the samples table through a foreign key.

In [14]:
# checking where each column would best fit

for col in ["age", "sex", "condition"]:
    print(
        col,
        df.groupby("subject")[col]
          .nunique()
          .gt(1)
          .sum()
    )

age 0
sex 0
condition 0


As seen above, the age, sex and condition are all unique to the subject and therefore it is suitable to place them in a table together.